# 使用 PolarDB-PG 作为 OpenAI 嵌入的向量数据库

本 notebook 将逐步指导你使用 PolarDB-PG 作为 OpenAI 嵌入的向量数据库。

本 notebook 展示了一个端到端的流程：
1. 使用由 OpenAI API 预先计算好的嵌入。
2. 将嵌入存储到 PolarDB-PG 云实例中。
3. 使用 OpenAI API 将原始文本查询转换为嵌入。
4. 使用 PolarDB-PG 在创建的集合中执行最近邻搜索。

### 什么是 PolarDB-PG

[PolarDB-PG](https://www.alibabacloud.com/help/en/polardb/latest/what-is-polardb-2) 是一个采用读写分离架构的高性能向量数据库。它是由阿里云托管的云原生数据库，100% 兼容 PostgreSQL，并高度兼容 Oracle 语法。它支持海量向量数据的存储与查询，并通过优化底层执行算法大幅提升向量计算效率，为用户提供快速、弹性、高性能、海量存储且安全可靠的向量数据库服务。此外，PolarDB-PG 还支持多维、多模态的时空信息引擎和地理信息引擎。同时，PolarDB-PG 具备完整的 OLAP 功能和服务等级协议，已获得众多用户的认可和使用；







### 部署选项

- 使用 [PolarDB-PG 云向量数据库](https://www.alibabacloud.com/product/polardb-for-postgresql)。[点击这里](https://www.alibabacloud.com/product/polardb-for-postgresql?spm=a3c0i.147400.6791778070.243.9f204881g5cjpP)快速部署。

## 前提条件

为完成本练习，我们需要准备以下几样东西：

1. PolarDB-PG 云服务器实例。
2. 用于与向量数据库交互的 'psycopg2' 库。任何其他 postgresql 客户端库也都可以。
3. 一个 [OpenAI API 密钥](https://beta.openai.com/account/api-keys)。

我们可以运行一个简单的 curl 命令来验证服务器是否已成功启动：

### 安装依赖

本 notebook 显然需要 `openai` 和 `psycopg2` 包，但我们还会用到其他一些库。以下命令会安装所有这些库：


In [ ]:
! pip install openai psycopg2 pandas wget

准备 OpenAI API 密钥
OpenAI API 密钥用于对文档和查询进行向量化。

如果你还没有 OpenAI API 密钥，可以从以下地址获取：https://beta.openai.com/account/api-keys.

拿到密钥后，请将其添加到环境变量 OPENAI_API_KEY 中。

如果你对通过环境变量设置 API 密钥有任何疑问，请参阅 [API 密钥安全最佳实践](https://help.openai.com/en/articles/5112595-best-practices-for-api-key-safety)。

In [3]:
# Test that your OpenAI API key is correctly set as an environment variable
# Note. if you run this notebook locally, you will need to reload your terminal and the notebook for the env variables to be live.

if os.getenv("OPENAI_API_KEY") is not None:
    print("OPENAI_API_KEY is ready")
else:
    print("OPENAI_API_KEY environment variable not found")

OPENAI_API_KEY is ready


## 连接 PolarDB
首先将其添加到环境变量中，或者直接修改下面 "psycopg2.connect" 的参数。

使用官方 Python 库连接正在运行的 PolarDB 服务器实例非常简单：

In [4]:
import os
import psycopg2

# Note. alternatively you can set a temporary env variable like this:
# os.environ["PGHOST"] = "your_host"
# os.environ["PGPORT"] "5432"),
# os.environ["PGDATABASE"] "postgres"),
# os.environ["PGUSER"] "user"),
# os.environ["PGPASSWORD"] "password"),

connection = psycopg2.connect(
    host=os.environ.get("PGHOST", "localhost"),
    port=os.environ.get("PGPORT", "5432"),
    database=os.environ.get("PGDATABASE", "postgres"),
    user=os.environ.get("PGUSER", "user"),
    password=os.environ.get("PGPASSWORD", "password")
)

# Create a new cursor object
cursor = connection.cursor()

我们可以运行任意可用的方法来测试连接：

In [5]:
# Execute a simple query to test the connection
cursor.execute("SELECT 1;")
result = cursor.fetchone()

# Check the query result
if result == (1,):
    print("Connection successful!")
else:
    print("Connection failed.")

Connection successful!


In [7]:
import wget

embeddings_url = "https://cdn.openai.com/API/examples/data/vector_database_wikipedia_articles_embedded.zip"

# The file is ~700 MB so this will take some time
wget.download(embeddings_url)

'vector_database_wikipedia_articles_embedded.zip'

然后需要解压下载的文件：

In [8]:
import zipfile
import os
import re
import tempfile

current_directory = os.getcwd()
zip_file_path = os.path.join(current_directory, "vector_database_wikipedia_articles_embedded.zip")
output_directory = os.path.join(current_directory, "../../data")

with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
    zip_ref.extractall(output_directory)


# check the csv file exist
file_name = "vector_database_wikipedia_articles_embedded.csv"
data_directory = os.path.join(current_directory, "../../data")
file_path = os.path.join(data_directory, file_name)


if os.path.exists(file_path):
    print(f"The file {file_name} exists in the data directory.")
else:
    print(f"The file {file_name} does not exist in the data directory.")

The file vector_database_wikipedia_articles_embedded.csv exists in the data directory.


## 索引数据

PolarDB 将数据存储在 __relation__ 中，其中每个对象至少由一个向量描述。我们的 relation 名为 **articles**，每个对象由 **title** 和 **content** 两个向量描述。

我们先创建 relation，并在 **title** 和 **content** 上分别创建向量索引，然后用预先计算好的嵌入填充它。

In [6]:
create_table_sql = '''
CREATE TABLE IF NOT EXISTS public.articles (
    id INTEGER NOT NULL,
    url TEXT,
    title TEXT,
    content TEXT,
    title_vector vector(1536),
    content_vector vector(1536),
    vector_id INTEGER
);

ALTER TABLE public.articles ADD PRIMARY KEY (id);
'''

# SQL statement for creating indexes
create_indexes_sql = '''
CREATE INDEX ON public.articles USING ivfflat (content_vector) WITH (lists = 1000);

CREATE INDEX ON public.articles USING ivfflat (title_vector) WITH (lists = 1000);
'''

# Execute the SQL statements
cursor.execute(create_table_sql)
cursor.execute(create_indexes_sql)

# Commit the changes
connection.commit()

## 加载数据

在本节中，我们将加载事先准备好的数据，这样你就不必使用自己的额度重新计算 Wikipedia 文章的嵌入。

In [ ]:
import io

# Path to your local CSV file
csv_file_path = '../../data/vector_database_wikipedia_articles_embedded.csv'

# Define a generator function to process the file line by line
def process_file(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            yield line

# Create a StringIO object to store the modified lines
modified_lines = io.StringIO(''.join(list(process_file(csv_file_path))))

# Create the COPY command for the copy_expert method
copy_command = '''
COPY public.articles (id, url, title, content, title_vector, content_vector, vector_id)
FROM STDIN WITH (FORMAT CSV, HEADER true, DELIMITER ',');
'''

# Execute the COPY command using the copy_expert method
cursor.copy_expert(copy_command, modified_lines)

# Commit the changes
connection.commit()

In [9]:
# Check the collection size to make sure all the points have been stored
count_sql = """select count(*) from public.articles;"""
cursor.execute(count_sql)
result = cursor.fetchone()
print(f"Count:{result[0]}")

Count:25000


## 搜索数据

数据放入 Qdrant 后，我们开始在集合中查询最接近的向量。可以额外提供参数 `vector_name`，在基于标题和基于内容的搜索之间切换。由于预计算的嵌入是使用 OpenAI 的 `text-embedding-3-small` 模型创建的，因此在搜索时也必须使用该模型。

In [10]:
def query_polardb(query, collection_name, vector_name="title_vector", top_k=20):

    # Creates embedding vector from user query
    embedded_query = openai.Embedding.create(
        input=query,
        model="text-embedding-3-small",
    )["data"][0]["embedding"]

    # Convert the embedded_query to PostgreSQL compatible format
    embedded_query_pg = "[" + ",".join(map(str, embedded_query)) + "]"

    # Create SQL query
    query_sql = f"""
    SELECT id, url, title, l2_distance({vector_name},'{embedded_query_pg}'::VECTOR(1536)) AS similarity
    FROM {collection_name}
    ORDER BY {vector_name} <-> '{embedded_query_pg}'::VECTOR(1536)
    LIMIT {top_k};
    """
    # Execute the query
    cursor.execute(query_sql)
    results = cursor.fetchall()

    return results

In [11]:
import openai

query_results = query_polardb("modern art in Europe", "Articles")
for i, result in enumerate(query_results):
    print(f"{i + 1}. {result[2]} (Score: {round(1 - result[3], 3)})")

1. Museum of Modern Art (Score: 0.5)
2. Western Europe (Score: 0.485)
3. Renaissance art (Score: 0.479)
4. Pop art (Score: 0.472)
5. Northern Europe (Score: 0.461)
6. Hellenistic art (Score: 0.457)
7. Modernist literature (Score: 0.447)
8. Art film (Score: 0.44)
9. Central Europe (Score: 0.439)
10. European (Score: 0.437)
11. Art (Score: 0.437)
12. Byzantine art (Score: 0.436)
13. Postmodernism (Score: 0.434)
14. Eastern Europe (Score: 0.433)
15. Europe (Score: 0.433)
16. Cubism (Score: 0.432)
17. Impressionism (Score: 0.432)
18. Bauhaus (Score: 0.431)
19. Surrealism (Score: 0.429)
20. Expressionism (Score: 0.429)


In [12]:
# This time we'll query using content vector
query_results = query_polardb("Famous battles in Scottish history", "Articles", "content_vector")
for i, result in enumerate(query_results):
    print(f"{i + 1}. {result[2]} (Score: {round(1 - result[3], 3)})")

1. Battle of Bannockburn (Score: 0.489)
2. Wars of Scottish Independence (Score: 0.474)
3. 1651 (Score: 0.457)
4. First War of Scottish Independence (Score: 0.452)
5. Robert I of Scotland (Score: 0.445)
6. 841 (Score: 0.441)
7. 1716 (Score: 0.441)
8. 1314 (Score: 0.429)
9. 1263 (Score: 0.428)
10. William Wallace (Score: 0.426)
11. Stirling (Score: 0.419)
12. 1306 (Score: 0.419)
13. 1746 (Score: 0.418)
14. 1040s (Score: 0.414)
15. 1106 (Score: 0.412)
16. 1304 (Score: 0.411)
17. David II of Scotland (Score: 0.408)
18. Braveheart (Score: 0.407)
19. 1124 (Score: 0.406)
20. July 27 (Score: 0.405)
